In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import numpy as np
import os

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

print(f"Dataset contents: {os.listdir(path)}")

if os.path.exists(os.path.join(path, "train")):
    train_dir = os.path.join(path, "train")
    test_dir = os.path.join(path, "test") if os.path.exists(os.path.join(path, "test")) else os.path.join(path, "val")
else:
    subdirs = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
    if len(subdirs) > 0:
        train_dir = os.path.join(path, subdirs[0])
        if len(subdirs) > 1:
            test_dir = os.path.join(path, subdirs[1])
        else:
            test_dir = train_dir
    else:
        train_dir = path
        test_dir = path

print(f"Using train_dir: {train_dir}")
print(f"Using test_dir: {test_dir}")

full_dataset = ImageFolder(train_dir, transform=transform)

if train_dir == test_dir:
    from torch.utils.data import random_split
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
else:
    train_dataset = full_dataset
    test_dataset = ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

fig, axes = plt.subplots(1, 5, figsize=(15, 5))

for i in range(5):
    if hasattr(train_dataset, 'dataset'):
        img, label = train_dataset.dataset[train_dataset.indices[i]]
        class_name = train_dataset.dataset.classes[label]
    else:
        img, label = train_dataset[i]
        class_name = train_dataset.classes[label]

    img_np = img.numpy().transpose(1, 2, 0)

    axes[i].imshow(img_np)
    axes[i].set_title(f"Label: {class_name}")
    axes[i].axis('off')

plt.show()

In [ ]:
# Write your code here
class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 2 * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN().to(device)
print(model)

In [ ]:
# Write your code here
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 5

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Train Accuracy={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Accuracy={val_acc:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
ax1.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
ax1.set_xlabel("Epochs")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curve")
ax1.legend()

ax2.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
ax2.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
ax2.set_xlabel("Epochs")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy Curve")
ax2.legend()

plt.show()

In [ ]:
# Write your code here
class PotatoCNN_Residual(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(192, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.conv5 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
        )

        self.skip_pool = nn.MaxPool2d(2)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 2 * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        x2_pooled = self.skip_pool(x2)
        x_concat = torch.cat([x3, x2_pooled], dim=1)

        x4 = self.conv4(x_concat)
        x5 = self.conv5(x4)

        x = self.classifier(x5)
        return x

model_res = PotatoCNN_Residual().to(device)
criterion_res = nn.CrossEntropyLoss()
optimizer_res = optim.Adam(model_res.parameters(), lr=0.001)

train_losses_res = []
train_accuracies_res = []
val_losses_res = []
val_accuracies_res = []


for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model_res, train_loader, criterion_res, optimizer_res, device)
    val_loss, val_acc = validate(model_res, test_loader, criterion_res, device)

    train_losses_res.append(train_loss)
    train_accuracies_res.append(train_acc)
    val_losses_res.append(val_loss)
    val_accuracies_res.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Train Accuracy={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Accuracy={val_acc:.2f}%")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(range(1, num_epochs+1), train_losses_res, label="Train Loss", marker='o')
ax1.plot(range(1, num_epochs+1), val_losses_res, label="Validation Loss", marker='o')
ax1.set_xlabel("Epochs")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curve (Residual Model)")
ax1.legend()

ax2.plot(range(1, num_epochs+1), train_accuracies_res, label="Train Accuracy", marker='o')
ax2.plot(range(1, num_epochs+1), val_accuracies_res, label="Validation Accuracy", marker='o')
ax2.set_xlabel("Epochs")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy Curve (Residual Model)")
ax2.legend()

plt.show()